# KD Comparison Tool
BLI / SPR Kinetic Result Table Analyzer

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

## Settings
Change these parameters as needed:

In [ ]:
# === SETTINGS ===
CSV_PATH = "2026_02_12_ Kinetic Result Table_local_fit_2_1_fab22_WT_all_Antigen.csv"
R2_THRESHOLD = 0.95
FIT_TYPE_OVERRIDE = None  # Set to "KD2" or "KD (M)" to override auto-detection

In [ ]:
# --- Helper functions ---

FIT_TYPE_MAP = {"fit_2_1": "KD2", "fit_1_1": "KD (M)"}
UNIT_THRESHOLDS = [
    (1e-9, "pM", 1e12),
    (1e-6, "nM", 1e9),
    (1.0,  "\u00b5M", 1e6),
]

def format_kd(value_m):
    if value_m is None or np.isnan(value_m):
        return "N/A"
    for threshold, unit, scale in UNIT_THRESHOLDS:
        if value_m < threshold:
            return f"{value_m * scale:.3g} {unit}"
    return f"{value_m * 1e6:.3g} \u00b5M"

def detect_fit_type(filename):
    name_lower = filename.lower()
    for pattern, col in FIT_TYPE_MAP.items():
        if pattern in name_lower:
            return col
    return None

def parse_kd_value(raw):
    if raw is None:
        return None, False
    s = str(raw).strip()
    if s == "" or s.lower() in ("nan", "n/a", "na", "#num!", "error"):
        return None, False
    below_limit = s.startswith("<")
    s_clean = s.lstrip("<").strip()
    try:
        return float(s_clean), below_limit
    except ValueError:
        return None, False

## Load & Clean Data

In [ ]:
# Determine KD column
if FIT_TYPE_OVERRIDE:
    kd_column = FIT_TYPE_OVERRIDE
    fit_source = "user override"
else:
    kd_column = detect_fit_type(CSV_PATH)
    if kd_column is None:
        raise ValueError(f"Could not auto-detect fit type from filename '{CSV_PATH}'. Set FIT_TYPE_OVERRIDE.")
    fit_source = f"auto-detected ({'fit_2_1' if kd_column == 'KD2' else 'fit_1_1'})"

print(f"KD column: {kd_column} ({fit_source})")
print(f"R\u00b2 threshold: {R2_THRESHOLD}")

# Read CSV
df = pd.read_csv(CSV_PATH, dtype=str)
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")]
print(f"\nLoaded {len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Clean data
stats = {"rows_raw": len(df)}

for col in ("Sample ID", "Loading Sample ID", "Conc. (nM)", "Full R^2", kd_column):
    df[col] = df[col].astype(str).str.strip()

# Filter buffer rows
df = df[~(df["Sample ID"].str.lower() == "buffer") & ~(df["Loading Sample ID"].str.lower() == "buffer")].copy()
stats["rows_after_buffer"] = len(df)

# Filter N/A concentrations
df = df[~df["Conc. (nM)"].str.lower().isin(["n/a", "na", "nan", ""])].copy()
stats["rows_after_conc"] = len(df)

# Filter by R^2
df["_r2"] = pd.to_numeric(df["Full R^2"], errors="coerce")
mask_r2 = df["_r2"] >= R2_THRESHOLD
stats["rows_excluded_r2"] = int((~mask_r2).sum())
df = df[mask_r2].copy()

# Parse KD values
parsed = df[kd_column].apply(parse_kd_value)
df["kd_value"] = [v for v, _ in parsed]
df["below_limit"] = [b for _, b in parsed]

mask_kd_missing = df["kd_value"].isna()
stats["rows_excluded_kd_missing"] = int(mask_kd_missing.sum())
df = df[~mask_kd_missing].copy()
stats["rows_used"] = len(df)

print(f"Rows: {stats['rows_raw']} raw -> {stats['rows_used']} after filtering")
print(f"  Excluded by R\u00b2: {stats['rows_excluded_r2']}")
print(f"  Excluded by missing KD: {stats['rows_excluded_kd_missing']}")
n_below = int(df["below_limit"].sum())
if n_below > 0:
    print(f"  Below detection limit: {n_below} rows (flagged with *)")

## KD Summary (Geometric Mean)

In [ ]:
# Compute geometric mean KD per (Sample ID x Loading Sample ID)
records = []
for (sample, loading), grp in df.groupby(["Sample ID", "Loading Sample ID"], sort=True):
    vals = grp["kd_value"].dropna().values
    if len(vals) == 0:
        continue
    geomean = float(np.exp(np.mean(np.log(vals))))
    records.append({
        "antigen": sample,
        "antibody": loading,
        "geomean_kd_M": geomean,
        "kd_min_M": float(vals.min()),
        "kd_max_M": float(vals.max()),
        "n_points": len(vals),
        "any_below_limit": bool(grp["below_limit"].any()),
    })
summary = pd.DataFrame(records)

antigens = sorted(summary["antigen"].unique().tolist())
antibodies = sorted(summary["antibody"].unique().tolist())
print(f"Antigens ({len(antigens)}): {antigens}")
print(f"Antibodies ({len(antibodies)}): {antibodies}")

# Auto-detect WT reference
wt_ref = next((ab for ab in sorted(antibodies, key=str.lower) if "wt" in ab.lower()), antibodies[0])
print(f"\nWT reference: {wt_ref}")

In [ ]:
# KD pivot table
numeric_pivot = summary.pivot(index="antigen", columns="antibody", values="geomean_kd_M"
    ).reindex(index=antigens, columns=antibodies)

kd_pivot = numeric_pivot.map(format_kd)

# Mark below-limit cells
below_pivot = summary.pivot(index="antigen", columns="antibody", values="any_below_limit"
    ).reindex(index=antigens, columns=antibodies)
for col in kd_pivot.columns:
    if col in below_pivot.columns:
        mask = below_pivot[col].fillna(False).astype(bool)
        kd_pivot.loc[mask, col] = kd_pivot.loc[mask, col] + " *"

display(Markdown("### Geometric Mean KD per Antigen x Antibody"))
display(Markdown("*`*` = at least one value was below detection limit*"))
display(kd_pivot)

In [ ]:
# Number of points used
n_pivot = summary.pivot(index="antigen", columns="antibody", values="n_points"
    ).reindex(index=antigens, columns=antibodies)
display(Markdown("### Number of Valid Points Used"))
display(n_pivot)

## Fold Changes

In [ ]:
if len(antibodies) >= 2 and wt_ref in numeric_pivot.columns:
    wt_vals = numeric_pivot[wt_ref]
    non_wt = [ab for ab in antibodies if ab != wt_ref]
    fc_data = {}
    for ab in non_wt:
        fc_data[ab] = numeric_pivot[ab] / wt_vals
    fc_pivot = pd.DataFrame(fc_data, index=antigens)

    display(Markdown(f"### Fold Change vs. {wt_ref}"))
    display(Markdown("*> 1 = weaker binding, < 1 = tighter binding vs reference*"))
    display(fc_pivot.style.format("{:.2f}x", na_rep="N/A"))
else:
    print("Only one antibody found - fold change requires at least two.")
    fc_pivot = pd.DataFrame()

## Plots

In [ ]:
# Heatmap: Log2 fold change
if wt_ref in antibodies and len(antibodies) >= 2:
    wt_vals = numeric_pivot[wt_ref]
    log2fc_pivot = numeric_pivot.copy()
    for ab in antibodies:
        log2fc_pivot[ab] = np.log2(numeric_pivot[ab] / wt_vals)
    title = f"Log\u2082 Fold Change vs. {wt_ref}"
    colorbar_title = f"Log\u2082(FC vs {wt_ref})"
else:
    log2fc_pivot = np.log2(numeric_pivot * 1e9)
    title = "Log\u2082 KD (nM)"
    colorbar_title = "Log\u2082 KD (nM)"

text_vals = [[f"{v:.2f}" if not np.isnan(v) else "N/A" for v in row] for row in log2fc_pivot.values]

fig = go.Figure(go.Heatmap(
    z=log2fc_pivot.values.tolist(),
    x=list(log2fc_pivot.columns),
    y=list(log2fc_pivot.index),
    text=text_vals, texttemplate="%{text}",
    colorscale="RdBu_r", zmid=0,
    colorbar=dict(title=colorbar_title),
    hoverongaps=False,
))
fig.update_layout(title=title, xaxis_title="Antibody", yaxis_title="Antigen",
                  height=max(300, len(antigens) * 45 + 150))
fig.show()

In [ ]:
# Bar chart: KD in nM
plot_df = summary.copy()
plot_df["KD (nM)"] = plot_df["geomean_kd_M"] * 1e9

fig = px.bar(plot_df, x="antigen", y="KD (nM)", color="antibody", barmode="group",
             log_y=True, title="Geometric Mean KD by Antigen (log scale)",
             labels={"antigen": "Antigen", "KD (nM)": "KD (nM)", "antibody": "Antibody"})
fig.update_layout(xaxis_tickangle=-30, height=450)
fig.show()

In [ ]:
# Scatter: individual KD values per concentration
plot_df2 = df.copy()
plot_df2["KD (nM)"] = plot_df2["kd_value"] * 1e9
plot_df2["Conc_nM"] = pd.to_numeric(plot_df2["Conc. (nM)"], errors="coerce")

n_antigens = len(plot_df2["Sample ID"].unique())
cols = min(3, n_antigens)
rows = int(np.ceil(n_antigens / cols))

fig = px.scatter(plot_df2, x="Conc_nM", y="KD (nM)", color="Loading Sample ID",
                 facet_col="Sample ID", facet_col_wrap=cols, log_y=True, log_x=True,
                 title="KD Values per Concentration Point",
                 labels={"Conc_nM": "Conc. (nM)", "KD (nM)": "KD (nM)",
                         "Loading Sample ID": "Antibody", "Sample ID": "Antigen"})
fig.update_layout(height=max(350, rows * 280))
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

## Raw Filtered Data

In [ ]:
display_cols = ["Sample ID", "Loading Sample ID", "Conc. (nM)", kd_column, "kd_value", "below_limit", "Full R^2", "_r2"]
display_cols = [c for c in display_cols if c in df.columns]
col_rename = {kd_column: f"{kd_column} (raw)", "kd_value": "KD parsed (M)", "_r2": "R\u00b2"}
display(df[display_cols].rename(columns=col_rename))

## Per-Antigen Detailed Tables (Publication Quality)
Each table shows one antigen with all antibodies side by side, listing KD and R² at every dilution.

In [ ]:
# Publication-quality per-antigen tables: WT & Fab side by side at each dilution
from IPython.display import HTML

antigen_list = sorted(df["Sample ID"].unique())
antibody_list = sorted(df["Loading Sample ID"].unique())

# Sort so WT comes first
antibody_order = sorted(antibody_list, key=lambda x: (0 if "wt" in x.lower() else 1, x.lower()))

for antigen in antigen_list:
    ag_df = df[df["Sample ID"] == antigen].copy()
    ag_df["Conc_nM"] = pd.to_numeric(ag_df["Conc. (nM)"], errors="coerce")
    ag_df = ag_df.sort_values("Conc_nM", ascending=False)

    # Build side-by-side table: one row per dilution, columns grouped by antibody
    rows_out = []
    dilutions = sorted(ag_df["Conc_nM"].dropna().unique(), reverse=True)

    for conc in dilutions:
        row = {"Conc. (nM)": f"{conc:g}"}
        for ab in antibody_order:
            ab_conc = ag_df[(ag_df["Loading Sample ID"] == ab) & (ag_df["Conc_nM"] == conc)]
            if ab_conc.empty:
                row[f"{ab} | KD"] = "—"
                row[f"{ab} | R\u00b2"] = "—"
            else:
                # Take first match if duplicates
                r = ab_conc.iloc[0]
                kd_val = r["kd_value"]
                below = r["below_limit"]
                kd_str = format_kd(kd_val)
                if below:
                    kd_str = "<" + kd_str
                row[f"{ab} | KD"] = kd_str
                row[f"{ab} | R\u00b2"] = f"{r['_r2']:.4f}"
        rows_out.append(row)

    # Add geometric mean row
    mean_row = {"Conc. (nM)": "Geo. Mean"}
    for ab in antibody_order:
        ab_vals = ag_df[ag_df["Loading Sample ID"] == ab]["kd_value"].dropna().values
        if len(ab_vals) > 0:
            gm = float(np.exp(np.mean(np.log(ab_vals))))
            mean_row[f"{ab} | KD"] = format_kd(gm)
        else:
            mean_row[f"{ab} | KD"] = "—"
        mean_row[f"{ab} | R\u00b2"] = ""
    rows_out.append(mean_row)

    table_df = pd.DataFrame(rows_out)
    table_df = table_df.set_index("Conc. (nM)")

    # Style for publication
    display(Markdown(f"---\n### {antigen}"))

    styled = (
        table_df.style
        .set_properties(**{
            "text-align": "center",
            "font-size": "12px",
            "border": "1px solid #ccc",
            "padding": "6px 10px",
        })
        .set_table_styles([
            {"selector": "th", "props": [
                ("background-color", "#2c3e50"),
                ("color", "white"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("font-size", "12px"),
                ("padding", "8px 10px"),
                ("border", "1px solid #ccc"),
            ]},
            {"selector": "th.row_heading", "props": [
                ("background-color", "#ecf0f1"),
                ("color", "#2c3e50"),
                ("font-weight", "bold"),
            ]},
            {"selector": "tr:last-child td", "props": [
                ("font-weight", "bold"),
                ("background-color", "#f7f9f9"),
                ("border-top", "2px solid #2c3e50"),
            ]},
            {"selector": "caption", "props": [
                ("font-size", "14px"),
                ("font-weight", "bold"),
                ("text-align", "left"),
                ("padding", "8px 0"),
            ]},
        ])
        .set_caption(f"{antigen} — KD at each concentration")
    )
    display(styled)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
from pathlib import Path
import re

output_dir = Path("tables_png")
output_dir.mkdir(exist_ok=True)

antigen_list = sorted(df["Sample ID"].unique())
antibody_list = sorted(df["Loading Sample ID"].unique())
antibody_order = sorted(antibody_list, key=lambda x: (0 if "wt" in x.lower() else 1, x.lower()))

saved_files = []

for antigen in antigen_list:
    ag_df = df[df["Sample ID"] == antigen].copy()
    ag_df["Conc_nM"] = pd.to_numeric(ag_df["Conc. (nM)"], errors="coerce")
    dilutions = sorted(ag_df["Conc_nM"].dropna().unique(), reverse=True)

    # Build table data
    col_headers = ["Conc. (nM)"]
    for ab in antibody_order:
        col_headers += [f"{ab}\nKD", f"{ab}\nR\u00b2"]

    cell_text = []
    for conc in dilutions:
        row = [f"{conc:g}"]
        for ab in antibody_order:
            ab_conc = ag_df[(ag_df["Loading Sample ID"] == ab) & (ag_df["Conc_nM"] == conc)]
            if ab_conc.empty:
                row += ["\u2014", "\u2014"]
            else:
                r = ab_conc.iloc[0]
                kd_str = format_kd(r["kd_value"])
                if r["below_limit"]:
                    kd_str = "<" + kd_str
                row += [kd_str, f"{r['_r2']:.4f}"]
        cell_text.append(row)

    # Geo mean row
    mean_row = ["Geo. Mean"]
    for ab in antibody_order:
        ab_vals = ag_df[ag_df["Loading Sample ID"] == ab]["kd_value"].dropna().values
        if len(ab_vals) > 0:
            gm = float(np.exp(np.mean(np.log(ab_vals))))
            mean_row += [format_kd(gm), ""]
        else:
            mean_row += ["\u2014", ""]
    cell_text.append(mean_row)

    # Render with matplotlib
    n_rows = len(cell_text)
    n_cols = len(col_headers)
    fig_width = max(8, n_cols * 1.4)
    fig_height = max(2, (n_rows + 1) * 0.45 + 0.8)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")
    ax.set_title(f"{antigen}", fontsize=14, fontweight="bold", pad=12, loc="left")

    table = ax.table(
        cellText=cell_text,
        colLabels=col_headers,
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)

    # Style header
    for j in range(n_cols):
        cell = table[0, j]
        cell.set_facecolor("#2c3e50")
        cell.set_text_props(color="white", fontweight="bold", fontsize=9)
        cell.set_edgecolor("#cccccc")

    # Style data rows
    for i in range(1, n_rows + 1):
        for j in range(n_cols):
            cell = table[i, j]
            cell.set_edgecolor("#cccccc")
            if j == 0:
                cell.set_facecolor("#ecf0f1")
                cell.set_text_props(fontweight="bold")
            # Last row (geo mean) bold with top border
            if i == n_rows:
                cell.set_facecolor("#f7f9f9")
                cell.set_text_props(fontweight="bold")
                cell.set_edgecolor("#2c3e50")

    safe_name = re.sub(r'[^\w\-]', '_', antigen)
    filepath = output_dir / f"{safe_name}.png"
    fig.savefig(filepath, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    saved_files.append(str(filepath))

print(f"Saved {len(saved_files)} tables to '{output_dir}/':")
for f in saved_files:
    print(f"  {f}")

## Export Results

In [ ]:
# Build export CSV
export_rows = []
for antigen in antigens:
    row = {"Antigen": antigen}
    ag_data = summary[summary["antigen"] == antigen]
    wt_kd = None
    wt_row = ag_data[ag_data["antibody"] == wt_ref]
    if not wt_row.empty:
        wt_kd = wt_row.iloc[0]["geomean_kd_M"]
    for ab in antibodies:
        ab_row = ag_data[ag_data["antibody"] == ab]
        if ab_row.empty:
            row[f"KD_{ab}_(M)"] = np.nan
            row[f"KD_{ab}_(nM)"] = np.nan
        else:
            kd_m = ab_row.iloc[0]["geomean_kd_M"]
            row[f"KD_{ab}_(M)"] = kd_m
            row[f"KD_{ab}_(nM)"] = kd_m * 1e9
            row[f"n_points_{ab}"] = int(ab_row.iloc[0]["n_points"])
            if ab != wt_ref and wt_kd is not None:
                row[f"FoldChange_{ab}_vs_{wt_ref}"] = kd_m / wt_kd
    export_rows.append(row)

export_df = pd.DataFrame(export_rows)
export_df.to_csv("kd_comparison_output.csv", index=False)
print("Saved to kd_comparison_output.csv")
display(export_df)